# SOL/USD: calidad y antiguedad de la conversion
## tl;dr
Los 12 casos sinteticos verifican tres tasas utilizables y nueve observaciones desconocidas bajo los limites declarados. No son trades, muestras del operador ni evidencia de rentabilidad. El cambio corrige reutilizacion indefinida de FX; no completa la alineacion entre marcas de runners y caja cotizada.

## Context & Methods
Grano: un caso de contrato de respuesta/tiempo, un activo (SOL), una divisa (USD). Metodos: parser real `utils/sol_price.py` y verificador real `fresh_sol_usd`, sin HTTP, wallet, bot, SQL, entrenamiento ni activacion de modelos.
Fuentes: `utils/sol_price.py`, `trader/papertrading.py`, `run_bot.py`, `research_loop/runner_forward.py`, `research_loop/entry_gate_forward.py`, `tests/test_sol_usd_evidence.py`. Referencia primaria: [CoinGecko Simple Price](https://docs.coingecko.com/demo/reference/simple-price), campo `last_updated_at`.
### Key Assumptions
Reloj sintetico 1800000000 UNIX s; recibo maximo 60 s, dato de mercado maximo 120 s y tolerancia futura del proveedor 5 s. Estos son limites del cliente, no SLA del proveedor. Las fronteras se fijan localmente para reproducibilidad. No hay historia real ni muestreo de cartera: el archivo PAPER actual estaba ausente al inspeccionarlo y no se infiere ausencia de trades de esa ausencia.

In [1]:
from pathlib import Path
import hashlib, json
from unittest.mock import patch
import pandas as pd
from utils import sol_price as sol
source = Path('utils/sol_price.py').read_bytes().replace(b'\r\n', b'\n')
print('contract_source_sha256=' + hashlib.sha256(source).hexdigest())
T0 = 1800000000.0
def body(price=100.0, updated=T0):
    return json.dumps({'solana': {'usd': price, 'last_updated_at': updated}}).encode()

contract_source_sha256=e22a4957e703cd523995190c98683466565476489bfd27b5ad952821b0973aad


## Data
Poblacion cerrada de casos sinteticos, no observaciones financieras. Cada cuerpo original y cada instante de evaluacion se mantienen explicitos. No se convierten desconocidos a cero; no hay uniones, duplicados de casos ni predictores de aprendizaje.

In [2]:
cases = [
    ('Fresh point', body(), T0, T0, 100.0),
    ('Market age exactly 120 s', body(updated=T0-120), T0, T0, 100.0),
    ('Market age 121 s', body(updated=T0-121), T0, T0, None),
    ('Client receipt age 61 s', body(), T0, T0+61, None),
    ('Market expires while receipt remains young', body(updated=T0-110), T0, T0+11, None),
    ('Provider time 6 s in future', body(updated=T0+6), T0, T0, None),
    ('String USD value', body(price='100'), T0, T0, None),
    ('Boolean USD value', body(price=True), T0, T0, None),
    ('Missing provider timestamp', b'{"solana":{"usd":100}}', T0, T0, None),
    ('Conflicting duplicate USD key', b'{"solana":{"usd":100,"usd":200,"last_updated_at":1800000000}}', T0, T0, None),
    ('Wrong requested asset', b'{"bitcoin":{"usd":100,"last_updated_at":1800000000}}', T0, T0, None),
    ('Provider future tolerance exactly 5 s', body(updated=T0+5), T0, T0, 100.0),
]
assert len({case[0] for case in cases}) == len(cases) == 12

## Results
Se evalua por separado el estado de parseo y la utilizabilidad posterior. Un punto valido al recibirse puede caducar despues. Una tabla de contrato expresa estas fronteras directamente; no se agrega un grafico porque no hay una tendencia ni una comparacion financiera observada.

In [3]:
rows = []
with patch.multiple(sol, _TTL_OK=60.0, _MAX_MARKET_AGE_S=120.0):
    for label, raw, received, evaluated, expected in cases:
        observation = sol.parse_sol_usd_body(raw, received_at=received)
        usable = sol.fresh_sol_usd(observation, now=evaluated)
        assert usable == expected, label
        rows.append({'Case': label, 'At receipt': observation.status,
                     'Rate USD/SOL': usable, 'Matches contract': usable == expected})
result = pd.DataFrame(rows)
display(result.fillna({'Rate USD/SOL': 'Unknown'}))

,Case,At receipt,Rate USD/SOL,Matches contract
0,Fresh point,OK,100.0,True
1,Market age exactly 120 s,OK,100.0,True
2,Market age 121 s,ERR,Unknown,True
3,Client receipt age 61 s,OK,Unknown,True
4,Market expires while receipt remains young,OK,Unknown,True
5,Provider time 6 s in future,ERR,Unknown,True
6,String USD value,ERR,Unknown,True
7,Boolean USD value,ERR,Unknown,True
8,Missing provider timestamp,ERR,Unknown,True
9,Conflicting duplicate USD key,ERR,Unknown,True


In [4]:
summary = {
    'synthetic_contract_cases': len(result),
    'usable_rates': int(result['Rate USD/SOL'].notna().sum()),
    'unknown_rates': int(result['Rate USD/SOL'].isna().sum()),
}
assert summary == {'synthetic_contract_cases': 12, 'usable_rates': 3, 'unknown_rates': 9}
assert result['Matches contract'].all()
assumed = sol.SolUsdObservation('ASSUMED', 123.0, source='configured_override', assumed=True)
assert sol.fresh_sol_usd(assumed, now=T0) is None
print(json.dumps(summary, sort_keys=True))

{"synthetic_contract_cases": 12, "unknown_rates": 9, "usable_rates": 3}


## Takeaways
La conversion solo es utilizable con precio positivo numerico y dos relojes aceptados; el cache no renueva el recibo original y un fallo no autoriza el ultimo dato bueno. El importe de entrada historico debe permanecer desconocido si falta el original, aunque el cambio actual este disponible. Los tests de integracion separados cubren el segundo chequeo tras la cotizacion y la preservacion de caja/cantidad ante FX desconocido.
Limites: este notebook prueba un contrato de software. No consulta credenciales ni accesibilidad real de CoinGecko, no valida ejecucion de swaps, no reconstruye el pasado y no demuestra beneficio futuro. Quedan pendientes las marcas financieras de runners ligadas a cantidad/caja, los recibos duraderos de valoracion y la aceptacion prospectiva de la estrategia completa.